## Checking Differentiability of Affine Transformation Functions in `PolySlab`.

 This notebook verifies whether the **affine transformation functions** 
(`translated`, `scaled`, `rotated`) in the `PolySlab` class are differentiable 
 using **Autograd**.


In [1]:
import tidy3d.components.geometry.polyslab as polyslab
from autograd.test_util import check_grads
import numpy as np

# Creates an instance of the `PolySlab` class, which represents an extruded 
# polygon with specified vertices, an extrusion axis, and slab bounds.

In [2]:
vertices = np.array([[0.0, 0.0],
                    [1.0, 0.0],
                    [1.0, 1.0]])
axis = 1 # 0 for x, 1 for y, 2 for z
slab_bounds = (-1.0, 1.0)

poly = polyslab.PolySlab(vertices=vertices,
                        axis=axis, 
                        slab_bounds=slab_bounds)

# Check type of vertices array

In [ ]:
x, y, z = 0.1, 0.2, 0.3  #Tanslation/ Scaling values
theta = np.pi/5.
new_poly_trans = poly.translated(x, y, z)
new_poly_scal = poly.scaled(x, y, z)
new_poly_rot = poly.rotated(theta,
                            axis)
print(type(new_poly_trans.vertices))
print(type(new_poly_scal.vertices))
print(type(new_poly_rot))

# Test autograd compatibility

In [4]:
def test_translation_grad(x: float, 
                          y: float, 
                          z: float
                          ) -> np.ndarray:
  """
  Computes the translated vertices of a `PolySlab` object.

  Parameters
  ----------
  x : float
      Translation distance along the x-axis.
  y : float
      Translation distance along the y-axis.
  z : float
      Translation distance along the z-axis.

  Returns
  -------
  np.ndarray
      The translated vertices of the `PolySlab` object.
  """
  new_poly = poly.translated(x, y, z)
  return new_poly.vertices


def test_scaling_grad(x: float,
                      y: float,
                      z: float
                      ) -> np.ndarray:
  """
    Computes the scaled vertices of a `PolySlab` object.

    Parameters
    ----------
    x : float
        Scaling factor along the x-axis.
    y : float
        Scaling factor along the y-axis.
    z : float
        Scaling factor along the z-axis.

    Returns
    -------
    np.ndarray
        The scaled vertices of the `PolySlab` object.
  """
  new_poly = poly.scaled(x, y, z)
  return new_poly.vertices


def test_rotation_grad(angle: float,
                       axis: int) -> np.ndarray:
  """
    Computes the rotated vertices of a `PolySlab` object.

    Parameters
    ----------
    angle : float
        Rotation angle in radians.
    axis : int
        Axis of rotation: 0 for x, 1 for y, 2 for z.

    Returns
    -------
    np.ndarray
        The rotated vertices of the `PolySlab` object.
  """
  if axis == poly.axis:
    new_poly = poly.rotated(angle, axis=axis)
  else:
    raise ValueError("Rotation for axes other than the PolySlab axis results in a non-differentiable Transformed object.")
  return new_poly.vertices

# Vertices is a NumPy array, but check_grads still works because the transformation
# parameters (x, y, z, theta) are differentiable (of type autograd.TracedVertices)

In [ ]:
#Translation
try:
    check_grads(lambda x: test_translation_grad(x, y, z), modes=['rev'])(x)
    print("Translation gradient check passed for x.")
except Exception as e:
    print("Translation gradient check failed for x:", e)

try:
    check_grads(lambda y: test_translation_grad(x, y, z), modes=['rev'])(y)
    print("Translation gradient check passed for y.")
except Exception as e:
    print("Translation gradient check failed for y:", e)

try:
    check_grads(lambda z: test_translation_grad(x, y, z), modes=['rev'])(z)
    print("Translation gradient check passed for z.")
except Exception as e:
    print("Translation gradient check failed for z:", e)

In [ ]:
#Scaling
try:
    check_grads(lambda x: test_scaling_grad(x, y, z), modes=['rev'])(x)
    print("Scaling gradient check passed for x.")
except Exception as e:
    print("Scaling gradient check failed for x:", e)

try:
    check_grads(lambda y: test_scaling_grad(x, y, z), modes=['rev'])(y)
    print("Scaling gradient check passed for y.")
except Exception as e:
    print("Scaling gradient check failed for y:", e)

try:
    check_grads(lambda z: test_scaling_grad(x, y, z), modes=['rev'])(z)
    print("Scaling gradient check passed for z.")
except Exception as e:
    print("Scaling gradient check failed for z:", e)


In [ ]:
#Rotation
try:
    check_grads(lambda theta: test_rotation_grad(theta, poly.axis),
                                                          modes=['rev'])(theta)
    print("Rotation gradient check passed for theta.")
except Exception as e:
    print("Rotation gradient check failed for theta:", e)